In [1]:
import os
import torch
from PIL import Image
import json

class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, root, transforms=None):
        self.root = root
        self.transforms = transforms
        with open(os.path.join(root, "annotations.json")) as f:
            self.annotations = json.load(f)

    def __len__(self):
        return len(self.annotations["images"])

    def __getitem__(self, idx):
        img_id = self.annotations["images"][idx]["id"]
        img_path = os.path.join(self.root, "images", self.annotations["images"][idx]["file_name"])
        img = Image.open(img_path).convert("RGB")

        # Get annotations
        annotations = [a for a in self.annotations["annotations"] if a["image_id"] == img_id]
        boxes = [a["bbox"] for a in annotations]  # Format: [x, y, width, height]
        labels = [a["category_id"] for a in annotations]

        # Convert to tensors
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        # Target dictionary
        target = {"boxes": boxes, "labels": labels}

        if self.transforms:
            img, target = self.transforms(img, target)

        return img, target


In [ ]:
from torchvision.transforms import functional as F
from torchvision import transforms

# Transform function
class Transform:
    def __call__(self, image, target):
        image = F.to_tensor(image)
        return image, target

# Load dataset
from torch.utils.data import DataLoader
train_dataset = CustomDataset("dataset/train", transforms=Transform())
val_dataset = CustomDataset("dataset/val", transforms=Transform())

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))


In [ ]:
import torchvision

# Load pre-trained Faster R-CNN
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)

# Modify the classifier for custom dataset
num_classes = 2  # Change this to your number of classes (including background)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = torchvision.models.detection.FastRCNNPredictor(in_features, num_classes)

In [ ]:
import torch.optim as optim

# Define optimizer and learning rate scheduler
optimizer = optim.Adam(model.parameters(), lr=0.005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# Training loop
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0

    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Forward pass
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        epoch_loss += losses.item()

        # Backward pass
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

    # Step the scheduler
    lr_scheduler.step()

    print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}")